<a href="https://colab.research.google.com/github/xunan007/comp5511-lab/blob/main/Assignment_3_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 3: The Future Path (Neuro-Symbolic & Agents)
**Course:** COMP5511 - Artificial Intelligence Concepts

Welcome to your final assignment! Take a deep breath. While the code today might look a bit longer, the *concepts* are incredibly intuitive because we are teaching the AI to solve problems exactly like a human does.

### The Core Concept: Brains and Hands
So far in this course, we've looked at two very different types of AI:
1. **Symbolic AI (Assignment 1):** This AI has great "hands" (it follows strict, perfect rules) but no "brain" (it can't understand context or learn).
2. **Deep Learning & LLMs (Assignment 2):** This AI has an amazing "brain" (it understands language, context, and patterns) but no "hands" (it is trapped inside a computer, cannot browse the live internet, and cannot use a calculator).

**AI Agents** are the future. An Agent combines the two. It uses a Large Language Model (LLM) as its **Brain** to think about a problem, and we give it Python code as **Tools (Hands)** to interact with the real world.

In this 2-week assignment, you will complete an Agent in two stages by filling in the missing code.
1. **The Local Weather Advisor:** Teaching an LLM to read live data from the Hong Kong Observatory.
2. **The Fact-Checker:** Teaching the AI to "think out loud" before it searches Wikipedia.

In [ ]:
# RUN THIS CELL FIRST: It installs the tools we need and sets up our environment.
!pip install transformers accelerate wikipedia ipywidgets requests -q

import re
import wikipedia
import requests
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from ipywidgets import Layout, Button, VBox, HBox, Output, Text, HTML
from IPython.display import display, clear_output

print("✅ Libraries installed successfully!")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.9 MB/s eta 0:00:00
✅ Libraries installed successfully!


### Loading the "Brain": Qwen3-4B-Instruct
We are going to load a real Small Language Model (SLM) called **Qwen3-4B-Instruct**.
* **4B** means it has 4 Billion parameters (digital neurons).
* It is small enough to run on Colab's free GPU, but it has been specially trained ("Instruct") to follow formatting rules and act as an Agent.

In [ ]:
print("Downloading and loading Qwen3-4B-Instruct... (This may take 2-3 minutes)")

model_id = "Qwen/Qwen3-4B-Instruct-2507"

# 1. Load the Tokenizer (The Dictionary)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 2. Load the Model to the GPU
llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

def generate_llm(system_instructions, user_text, max_tokens=200):
    """
    This function talks to our Qwen model. We give it instructions (the rules)
    and the user's text, and it generates a response.
    (You do not need to edit this function).
    """
    messages = [
        {"role": "system", "content": system_instructions},
        {"role": "user", "content": user_text}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad():
        generated_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

print("✅ AI Brain loaded and ready!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

✅ AI Brain loaded and ready!


---
## Part 1: The Weather Advisor (Connecting to Live APIs)

### What is an API?
An API (Application Programming Interface) is how machines talk to other machines on the internet.
The **Hong Kong Observatory** provides a free open API. Our AI cannot browse the web, so we will write a Python function that uses the API to fetch live temperature data. Then, we will give this function to our AI as a "Tool".

In [ ]:
def get_hk_weather(district_name):
    """
    Our Tool: It reaches out to the HK Observatory API, downloads the JSON data,
    and searches for the temperature of the requested district.
    (You do not need to edit this function).
    """
    api_url = "https://data.weather.gov.hk/weatherAPI/opendata/weather.php?dataType=rhrread&lang=en"

    try:
        response = requests.get(api_url)
        data = response.json()

        temp_list = data.get("temperature", {}).get("data", [])

        for location in temp_list:
            if district_name.lower() in location["place"].lower():
                value = location["value"]
                unit = location["unit"]
                return f"Observation: The current temperature in {location['place']} is {value}°{unit}."

        return f"Error: Could not find temperature data for '{district_name}'."

    except Exception as e:
        return "Error: Failed to connect to the HK Observatory API."

print("Testing live HK Observatory Tool:")
print(get_hk_weather("Sha Tin"))

Testing live HK Observatory Tool:
Observation: The current temperature in Sha Tin is 27°C.


### Task 1.2: The Agent Loop (The Brain meets the Hands)
Now we build the "Agent Logic."

**How it works:**
1. We create an empty `context` variable (this acts as the AI's notepad).
2. We ask the AI to generate text based on the rules.
3. We use **Regular Expressions (`re.search`)** to scan the AI's text for specific commands.
4. If we find the AI asking for a tool, we run the Python tool, write the answer down in our `context` notepad, and the loop repeats.

**YOUR TASK:** Fill in the missing code sections marked with `___`.

In [ ]:
def weather_agent_loop(user_query, max_steps=3):
    """
    This function manages the conversation. We use a 'for' loop to let the
    AI think and act up to 'max_steps' times before we force it to stop.
    """
    context = ""

    for step in range(1, max_steps + 1):

        # --- STUDENT TASK 1.2a: Write the System Instructions ---
        # The AI needs to know the rules. Tell it its role, the tool it has
        # (get_hk_weather), and the exact format it must use.
        # Format required: "Action: [tool name]" and "Action Input: [district]".
        # When finished, it must say "Final Answer: [advice]".
        system_instructions = f"""
        You are a helpful Hong Kong Weather Assistant.
        You have access to ONE tool:
        Tool Name: get_hk_weather

        # YOUR CODE HERE (Write the instructions explaining how to use the tool and format the answer)
        ___

        Context (What we know so far):
        {context}
        """

        print(f"\n--- STEP {step} ---")
        print("🤖 AI is thinking...")
        llm_output = generate_llm(system_instructions, user_query)
        print(f"💬 AI Output:\n{llm_output}\n")

        # The "Parser" scans the text for commands
        action_match = re.search(r"Action:\s*(.*)", llm_output, re.IGNORECASE)
        input_match = re.search(r"Action Input:\s*(.*)", llm_output, re.IGNORECASE)
        final_answer_match = re.search(r"Final Answer:\s*(.*)", llm_output, re.DOTALL | re.IGNORECASE)

        if final_answer_match:
            # We found the answer! Break the loop.
            answer = final_answer_match.group(1).strip()
            print(f"✅ Agent finished task! Answer: {answer}")
            return answer

        elif action_match and input_match:
            # --- STUDENT TASK 1.2b: Extract the action and input ---
            # action_match.group(1) contains the raw string the AI generated.
            # Assign these to variables so we can use them. Don't forget to use .strip() to remove extra spaces!

            action = ___      # YOUR CODE HERE
            action_input = ___ # YOUR CODE HERE

            print(f"⚙️ We caught a tool request! Running python code: {action}('{action_input}')")

            if "get_hk_weather" in action.lower():
                tool_result = get_hk_weather(action_input)
                print(f"📥 Tool returned this data: {tool_result}")

                # --- STUDENT TASK 1.2c: Update the AI's memory ---
                # The AI needs to remember what the tool returned on the next loop.
                # Add the 'tool_result' to the 'context' string.

                context += ___ # YOUR CODE HERE

                print("🔄 Saving data to memory and looping back for the next step...")
            else:
                print("Error: The AI asked for a tool that doesn't exist.")
                return "Task failed."
        else:
            print("Error: The AI didn't format its answer correctly.")
            return "Task failed."

    print("🛑 Agent took too many steps! Stopping to prevent infinite loops.")
    return "Task failed: Reached maximum steps."

# Let's run the Agent!
print("Starting Weather Agent Loop...")
final_result = weather_agent_loop("I'm going to Sha Tin for a hike today. Should I wear a jacket?")

Starting Weather Agent Loop...

--- STEP 1 ---
🤖 AI is thinking...
💬 AI Output:
To help you decide whether to wear a jacket for your hike in Sha Tin, I'll check the current weather conditions in Sha Tin. This will include temperature and weather type (e.g., sunny, rainy, windy), which will guide your clothing choice.

I'll use the tool: get_hk_weather

Request: Get the current weather in Sha Tin.  
Answer format: Provide a clear recommendation based on the temperature and conditions. For example: "Yes, wear a jacket because the temperature is 10°C with light rain." or "No, you don't need a jacket because it's 25°C and sunny."

Let me get that information for you.  
{"tool": "get_hk_weather", "parameters": {"location": "Sha Tin"}}

Error: The AI didn't format its answer correctly.


---
## Part 2: The Wikipedia Fact-Checker (ReAct Pattern)

### What is "ReAct"? (Reasoning + Acting)
AI researchers discovered that LLMs get significantly smarter if you force them to "think out loud" before they act. This is called the **ReAct pattern**. We force the AI to output a `Thought:` before every `Action:`.

In [ ]:
def wikipedia_search(query):
    """Searches real Wikipedia and returns a short summary."""
    try:
        search_results = wikipedia.search(query, results=1)
        if not search_results:
            return "No Wikipedia page found for this query."
        page_title = search_results[0]
        summary = wikipedia.summary(page_title, sentences=2)
        return summary
    except wikipedia.exceptions.DisambiguationError as e:
        return "Query is too broad. Please be more specific."
    except Exception as e:
        return f"Wikipedia Error: {str(e)}"

### Task 2.2: The "Scratchpad" Memory & User Interface
Language Models have the memory of a goldfish. Every time you send a prompt, it forgets everything that happened previously.

To fix this, Agents use a **Scratchpad**. We create a giant text string that records everything that has happened so far.

**YOUR TASK:** Complete the ReAct Agent loop.

In [ ]:
def react_fact_checker(claim, max_iterations=4):
    """A complete Agent loop using the ReAct pattern."""

    system_prompt = """You are a rigorous Fact-Checking Agent. Verify the user's claim.
You have access to the following tool:
- wikipedia_search: Use this to search Wikipedia for facts.

You MUST use the following format to solve the problem step-by-step:
Thought: Think about what you need to do next.
Action: wikipedia_search
Action Input: the search query
Observation: (Wait for me to provide the result of the tool)
... (this cycle can repeat multiple times)
Thought: I now have enough information to make a decision.
Final Answer: True or False, followed by a brief explanation.
"""

    scratchpad = ""
    user_initial_prompt = f"Claim to verify: {claim}"

    for step in range(1, max_iterations + 1):

        # --- STUDENT TASK 2.2a: Give the AI its memory ---
        # The AI needs to see the user's prompt AND the history of what it has done so far.
        # Combine 'user_initial_prompt' and 'scratchpad' into a single string called current_query.
        current_query = ___ # YOUR CODE HERE

        llm_response = generate_llm(system_prompt, current_query, max_tokens=250)

        # Record the AI's response in our memory notebook
        scratchpad += llm_response + "\n"

        final_answer = re.search(r"Final Answer:\s*(.*)", llm_response, re.DOTALL | re.IGNORECASE)

        # --- STUDENT TASK 2.2b: Check for a final answer ---
        # If the regex found a final answer, we should return the scratchpad and the answer.
        # Check if 'final_answer' is not None.
        if ___: # YOUR CODE HERE
            return scratchpad, final_answer.group(1).strip()

        action_match = re.search(r"Action:\s*(.*)", llm_response, re.IGNORECASE)
        input_match = re.search(r"Action Input:\s*(.*)", llm_response, re.IGNORECASE)

        if action_match and input_match:
            action = action_match.group(1).strip()
            action_input = input_match.group(1).strip()

            if "wikipedia" in action.lower():
                observation = wikipedia_search(action_input)

                # --- STUDENT TASK 2.2c: Record the Observation ---
                # We ran the tool! Now we need to append the word "Observation: "
                # followed by the 'observation' variable to the scratchpad.
                # Don't forget to add a newline ("\n") at the end so it looks neat.
                scratchpad += ___ # YOUR CODE HERE
            else:
                scratchpad += "Observation: Tool not found.\n"
        else:
            return scratchpad, "Error: Agent forgot to use the proper formatting rules."

    return scratchpad, "Error: Agent took too many steps and got confused."


# --- Interactive Fact-Checker UI ---
out_react = Output()
claim_input = Text(placeholder="e.g., Hong Kong Disneyland is located on Hong Kong Island.", layout=Layout(width='450px'))
btn_fact_check = Button(description="Fact-Check!", button_style="info")

def run_fact_checker(_):
    with out_react:
        clear_output()
        print(f"🔍 Analyzing Claim: '{claim_input.value}'\n")
        print("=" * 50)

        scratchpad, final_answer = react_fact_checker(claim_input.value)

        print("🧠 READ THE AGENT'S MIND (Scratchpad History):")
        print(scratchpad)
        print("=" * 50)

        display(HTML(f"<div style='padding: 10px; background-color: #e6f3ff; border-radius: 5px; font-size: 16px;'><b>Final Verdict:</b> {final_answer}</div>"))

btn_fact_check.on_click(run_fact_checker)

display(VBox([
    HTML("<h3>The Wikipedia Fact Checker Agent</h3>"),
    HTML("<p>Type a claim below. Watch how the agent thinks step-by-step and uses Wikipedia to find the truth.</p>"),
    HBox([claim_input, btn_fact_check]),
    out_react
]))